# 已选股票池 Beta 分析（相对于沪深300）

本 notebook 针对 **已经选出的股票池**（四川成渝、新华文轩、绿色动力、昭衍新药、吉宏股份），
只做一件事：

- 用 Tushare Pro 日线数据，计算每只股票 **相对于沪深300（000300.SH）** 的 Beta；
- 计算两个窗口：近 3 年 & 近 120 交易日；
- 使用 Newey-West（HAC）标准误的 CAPM 回归，输出 Beta、标准误、p 值和 R²，方便你根据大盘节奏做择时和仓位调整。

字体设置和 Tushare 连接方式与 `sichuan_chengyu_beta_analysis.ipynb` 保持一致，以避免中文显示和连接问题。

In [1]:
import os
from pathlib import Path
import datetime as dt
from typing import Optional
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tushare as ts
from IPython.display import display
from statsmodels.api import OLS, add_constant

warnings.filterwarnings("ignore", message="verbose is deprecated", category=FutureWarning)
plt.style.use("seaborn-v0_8-darkgrid")
pd.set_option("display.float_format", "{:,.4f}".format)
sns.set_context("talk")

# 字体配置：尽量使用 Noto Sans CJK，回避中文缺字和 minus 号问题
from matplotlib import font_manager
font_paths = [
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc',
    '/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc',
]
for font_path in font_paths:
    if os.path.exists(font_path):
        try:
            font_manager.fontManager.addfont(font_path)
        except Exception:
            pass
try:
    font_manager._rebuild()
except AttributeError:
    pass
available_fonts = {f.name for f in font_manager.fontManager.ttflist}
preferred_fonts = [
    'Noto Sans CJK SC',
    'Noto Sans CJK JP',
    'SimHei',
    'Microsoft YaHei',
    'WenQuanYi Micro Hei',
    'Source Han Sans SC',
    'Droid Sans Fallback',
]
selected_font = None
for font in preferred_fonts:
    if font in available_fonts:
        selected_font = font
        break
plt.rcParams['font.family'] = 'sans-serif'
if selected_font:
    plt.rcParams['font.sans-serif'] = [selected_font, 'DejaVu Sans']
    print(f"Using font: {selected_font}")
else:
    plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    print('Using default DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False

Using font: Noto Sans CJK JP


In [2]:
# Tushare Pro 连接配置，与原 Beta notebook 一致
def load_token() -> Optional[str]:
    env_token = os.getenv('TUSHARE_API_KEY')
    if env_token:
        return env_token.strip()
    for candidate in [Path('.tushare_token'), Path.home() / '.tushare_token']:
        if candidate.is_file():
            token = candidate.read_text(encoding='utf-8').strip()
            if token:
                return token
    return None


def init_pro_client(base_url: Optional[str] = None):
    token = load_token()
    if not token:
        raise RuntimeError('未能读取 Tushare Token：请先配置 TUSHARE_API_KEY 或 .tushare_token')
    pro_client = ts.pro_api(token)
    if base_url:
        pro_client._DataApi__http_url = base_url.rstrip('/')
    return pro_client


BASE_URL = os.getenv('TUSHARE_BASE_URL') or 'http://api.tushare.pro/dataapi'
pro = init_pro_client(BASE_URL)
print(f"Base URL = {getattr(pro, '_DataApi__http_url', 'default')}")

Base URL = http://api.tushare.pro/dataapi


In [3]:
# 参数与股票池设置
MARKET_CODE = '000300.SH'  # 沪深300 作为市场组合
YEARS_LOOKBACK = 3
SHORT_WINDOW_DAYS = 120
HAC_LAGS = 5  # Newey-West 最大滞后

end_date = dt.date.today()
start_date = end_date - dt.timedelta(days=YEARS_LOOKBACK * 365)
START_DATE = start_date.strftime('%Y%m%d')
END_DATE = end_date.strftime('%Y%m%d')

STOCK_POOL = {
    '四川成渝': '601717.SH',
    '新华文轩': '601811.SH',
    '绿色动力': '601330.SH',
    '昭衍新药': '603127.SH',
    '吉宏股份': '002803.SZ',
}

print(f"分析区间: {START_DATE} ~ {END_DATE}")
print(f"市场指数: {MARKET_CODE}")


def fetch_pro_daily(ts_code: str, is_index: bool = False) -> pd.DataFrame:
    """从 Pro 获取收盘价时间序列（trade_date, close）。"""
    fields = 'trade_date,close'
    fetcher = pro.index_daily if is_index else pro.daily
    df = fetcher(ts_code=ts_code, start_date=START_DATE, end_date=END_DATE, fields=fields)
    if df.empty:
        raise RuntimeError(f'Pro 数据为空：{ts_code}，请检查权限或网络/权限。')
    df['trade_date'] = pd.to_datetime(df['trade_date'])
    df.sort_values('trade_date', inplace=True)
    return df


def prepare_returns(df: pd.DataFrame, label: str) -> pd.DataFrame:
    frame = df[['trade_date', 'close']].copy()
    frame.rename(columns={'close': f'close_{label}'}, inplace=True)
    frame[f'ret_{label}'] = frame[f'close_{label}'].pct_change()
    return frame


def slice_recent(df: pd.DataFrame, trading_days: int) -> pd.DataFrame:
    if trading_days >= len(df):
        return df.copy()
    return df.tail(trading_days).copy()


def capm_beta(window: pd.DataFrame):
    """在给定窗口上跑 CAPM：ret_stock ~ alpha + beta * ret_index，返回 beta 等指标。"""
    df = window.dropna(subset=['ret_stock', 'ret_index']).copy()
    if len(df) < 30:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    X = add_constant(df['ret_index'])
    model = OLS(df['ret_stock'], X).fit(cov_type='HAC', cov_kwds={'maxlags': HAC_LAGS})
    beta = model.params['ret_index']
    alpha = model.params['const']
    se = model.bse['ret_index']
    pval = model.pvalues['ret_index']
    r2 = model.rsquared
    return beta, alpha, se, pval, r2

分析区间: 20221211 ~ 20251210
市场指数: 000300.SH


In [4]:
# 统一获取指数数据（只取一次），然后对每只股票计算 3 年 & 120 日 Beta
index_raw = fetch_pro_daily(MARKET_CODE, is_index=True)
index_ret = prepare_returns(index_raw, 'index')

rows = []
for name, code in STOCK_POOL.items():
    stock_raw = fetch_pro_daily(code, is_index=False)
    stock_ret = prepare_returns(stock_raw, 'stock')

    combined = pd.merge(stock_ret, index_ret, on='trade_date', how='inner')
    combined.dropna(subset=['ret_stock', 'ret_index'], inplace=True)
    if combined.empty:
        print(f"{name} ({code}) 无重叠样本，跳过")
        continue

    window_3y = combined.copy()
    window_120 = slice_recent(combined, SHORT_WINDOW_DAYS)

    beta3, alpha3, se3, p3, r23 = capm_beta(window_3y)
    beta120, alpha120, se120, p120, r2120 = capm_beta(window_120)

    rows.append({
        'name': name,
        'ts_code': code,
        'n_3y': len(window_3y),
        'beta_3y': beta3,
        'beta_3y_se': se3,
        'beta_3y_p': p3,
        'r2_3y': r23,
        'n_120': len(window_120),
        'beta_120': beta120,
        'beta_120_se': se120,
        'beta_120_p': p120,
        'r2_120': r2120,
    })

summary = pd.DataFrame(rows)
summary = summary[['name', 'ts_code', 'n_3y', 'beta_3y', 'beta_3y_se', 'beta_3y_p', 'r2_3y',
                   'n_120', 'beta_120', 'beta_120_se', 'beta_120_p', 'r2_120']]

print("\nBeta 汇总（相对于沪深300 000300.SH）:")
print(summary.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

display(summary)


Beta 汇总（相对于沪深300 000300.SH）:
name   ts_code  n_3y  beta_3y  beta_3y_se  beta_3y_p  r2_3y  n_120  beta_120  beta_120_se  beta_120_p  r2_120
四川成渝 601717.SH   725   0.9594      0.0620     0.0000 0.2560    120    0.9873       0.1840      0.0000  0.1391
新华文轩 601811.SH   725   0.7015      0.0869     0.0000 0.1143    120    0.2767       0.1038      0.0077  0.0365
绿色动力 601330.SH   725   0.5898      0.0724     0.0000 0.2023    120    0.2100       0.1176      0.0743  0.0315
昭衍新药 603127.SH   725   1.3820      0.1358     0.0000 0.1954    120    1.2728       0.3493      0.0003  0.0861
吉宏股份 002803.SZ   725   1.0333      0.0947     0.0000 0.1241    120    0.2551       0.2844      0.3697  0.0085


,name,ts_code,n_3y,beta_3y,beta_3y_se,beta_3y_p,r2_3y,n_120,beta_120,beta_120_se,beta_120_p,r2_120
0,四川成渝,601717.SH,725,0.9594,0.0620,0.0000,0.2560,120,0.9873,0.1840,0.0000,0.1391
1,新华文轩,601811.SH,725,0.7015,0.0869,0.0000,0.1143,120,0.2767,0.1038,0.0077,0.0365
2,绿色动力,601330.SH,725,0.5898,0.0724,0.0000,0.2023,120,0.2100,0.1176,0.0743,0.0315
3,昭衍新药,603127.SH,725,1.3820,0.1358,0.0000,0.1954,120,1.2728,0.3493,0.0003,0.0861
4,吉宏股份,002803.SZ,725,1.0333,0.0947,0.0000,0.1241,120,0.2551,0.2844,0.3697,0.0085
